In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

from src.utils.logger import setup_logger
from src.utils.paths import RunPaths

PIPELINE_NAME = "ced-distill"
RUN_TRAINING = False
RESUME_RUN_ID = None

OUTPUT_DIR = ROOT / "output" if RUN_TRAINING else ROOT / "output" / "test"
run = (
    RunPaths.from_run_id(RESUME_RUN_ID, OUTPUT_DIR) if RESUME_RUN_ID else RunPaths.new(PIPELINE_NAME, OUTPUT_DIR)
).create()

logger = setup_logger(run.run_id, run.log_file)
logger.info("pipeline=%s | run=%s | mode=%s", PIPELINE_NAME, run.run_id, "TRAINING" if RUN_TRAINING else "SMOKE TEST")
logger.info(
    "log=%s | checkpoint=%s | result=%s | figure=%s", run.log_file, run.checkpoint_dir, run.result_dir, run.figure_dir
)

In [ ]:
from dataclasses import replace

import torch
import yaml

from src.train import CEDTrainConfig
from src.utils.seed import set_seed

with open(ROOT / "config" / "ced.yaml", encoding="utf-8") as _f:
    ced_yaml = yaml.safe_load(_f)
with open(ROOT / "config" / "beat.yaml", encoding="utf-8") as _f:
    beat_yaml = yaml.safe_load(_f)

ALL_SNR = (-5, 0, 5, 10, 15, 20)

CONFIG = {
    "seed": ced_yaml["train"]["seed"],
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "num_workers": 4,
    "resume": RESUME_RUN_ID is not None,
    "max_samples": None,
    "sample_rate": ced_yaml["frontend"]["sample_rate"],
    "clip_seconds": 4.0,
    "snr_groups": {
        "low_snr": (-5, 0),
        "mid_snr": (5, 10),
        "high_snr": (15, 20),
        "all_snr": ALL_SNR,
    },
    "ced": CEDTrainConfig.from_dict({**ced_yaml["train"], "kd_temperature": ced_yaml["loss"]["kd"]["temperature"]}),
}

if not RUN_TRAINING:
    CONFIG.update(max_samples=8, num_workers=0)
    CONFIG["ced"] = replace(CONFIG["ced"], epochs=1, batch_size=4, eval_batch_size=4)

DEVICE = torch.device(CONFIG["device"])
set_seed(CONFIG["seed"])
logger.info(
    "run_training=%s | device=%s | seed=%d | resume=%s | max_samples=%s",
    RUN_TRAINING,
    DEVICE,
    CONFIG["seed"],
    CONFIG["resume"],
    CONFIG["max_samples"],
)
logger.info(
    "snr_groups=%s | ced epochs=%d patience=%d lr=%.1e kd_temperature=%.1f",
    CONFIG["snr_groups"],
    CONFIG["ced"].epochs,
    CONFIG["ced"].patience,
    CONFIG["ced"].lr,
    CONFIG["ced"].kd_temperature,
)

In [ ]:
from scripts.download_dataset import ensure_dataset
from scripts.load_dataset import MIXTURE_KEY, Mixture36Dataset, build_dataloader, read_class_mids, subsample

ensure_dataset(MIXTURE_KEY, ROOT / "datasets")

max_samples = CONFIG["max_samples"]


def make_loader(dataset, batch_size, shuffle):
    return build_dataloader(dataset, batch_size, shuffle, CONFIG["num_workers"])


def make_ced_loaders(snr_levels):
    ced_config = CONFIG["ced"]
    datasets = {
        split: subsample(
            Mixture36Dataset(split=split, snr_db=tuple(float(s) for s in snr_levels), duration=CONFIG["clip_seconds"]),
            max_samples,
        )
        for split in ("train", "validation", "test")
    }
    return {
        "train": make_loader(datasets["train"], ced_config.batch_size, True),
        "validation": make_loader(datasets["validation"], ced_config.eval_batch_size, False),
        "test": make_loader(datasets["test"], ced_config.eval_batch_size, False),
    }


class_names = Mixture36Dataset(split="train").class_names
class_mids = read_class_mids(class_names)
logger.info("%s: %d classes mapped to AudioSet mids", MIXTURE_KEY, len(class_mids))

for _levels in CONFIG["snr_groups"].values():
    _sizes = {split: len(loader.dataset) for split, loader in make_ced_loaders(_levels).items()}
    logger.info("%s (ced) snr=%s: %s", MIXTURE_KEY, _levels, _sizes)

In [ ]:
from scripts.load_pretrain import PRETRAINED, download_pretrained

logger.info("downloading / locating pretrained checkpoints")
pretrained_paths = {"beat-teacher": download_pretrained("beat-teacher"), "ced": download_pretrained("ced")}
for _name, _path in pretrained_paths.items():
    logger.info("%s pretrain: %s (%.1f MB)", _name, _path.name, _path.stat().st_size / 1e6)

In [ ]:
teacher_spec, ced_spec = PRETRAINED["beat-teacher"], PRETRAINED["ced"]

teacher = teacher_spec.model_cls.from_yaml(teacher_spec.config_path)
teacher_report = teacher.load_pretrained(pretrained_paths["beat-teacher"])
teacher.select_classes(class_mids)
logger.info(
    "beat teacher: pretrained loaded | missing=%d unexpected=%d | classes=%d of 527",
    len(teacher_report.missing),
    len(teacher_report.unexpected),
    len(teacher.class_indices),
)

ced_pretrained = ced_spec.model_cls.from_yaml(ced_spec.config_path)
ced_report = ced_pretrained.load_pretrained(pretrained_paths["ced"])
logger.info(
    "ced student: pretrained loaded | missing=%d unexpected=%d", len(ced_report.missing), len(ced_report.unexpected)
)

ced_state = {key: value.clone() for key, value in ced_pretrained.state_dict().items()}


def new_ced():
    model = ced_spec.model_cls.from_yaml(ced_spec.config_path)
    model.load_state_dict(ced_state)
    return model


logger.info(
    "params: teacher=%.2fM | student=%.2fM", teacher.num_parameters() / 1e6, ced_pretrained.num_parameters() / 1e6
)

In [ ]:
import csv
import json

from tqdm.auto import tqdm

from src.train import CEDTrainer

metric_names = ["top1_accuracy", "top3_accuracy", "balanced_accuracy", "macro_f1", "mAP", "macro_auc"]
teacher_metric_names = ["top1_accuracy", "macro_f1", "mAP"]


def run_scenario(index, total, group, levels):
    name = f"{group}_nomask"
    result_path = run.history_path(name)
    if CONFIG["resume"] and result_path.is_file():
        logger.info("[%d/%d] %s: loaded cached result", index, total, name)
        return json.loads(result_path.read_text(encoding="utf-8"))

    logger.info("[%d/%d] %s: snr=%s", index, total, name, levels)
    set_seed(CONFIG["seed"])
    loaders = make_ced_loaders(levels)
    trainer = CEDTrainer(new_ced(), teacher, CONFIG["ced"], DEVICE, logger, name, run.checkpoint_path(name))
    fit = trainer.fit(loaders["train"], loaders["validation"])
    test = trainer.evaluate(loaders["test"], desc=f"{name} - test")
    teacher_test = trainer.evaluate_teacher(loaders["test"], desc=f"{name} - teacher test")
    result = {
        "scenario": name,
        "snr_group": group,
        "snr_levels": list(levels),
        "masking": False,
        "best_epoch": fit.best_epoch,
        "epochs_run": fit.epochs_run,
        "test": test,
        "teacher_test": teacher_test,
        "history": fit.history,
    }
    result_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    logger.info(
        "[%d/%d] %s: student top1=%.4f macro_f1=%.4f mAP=%.4f | teacher top1=%.4f macro_f1=%.4f mAP=%.4f",
        index,
        total,
        name,
        test["top1_accuracy"],
        test["macro_f1"],
        test["mAP"],
        teacher_test["top1_accuracy"],
        teacher_test["macro_f1"],
        teacher_test["mAP"],
    )
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result


def write_result_csv(rows):
    columns = [
        "scenario",
        "snr_group",
        "snr_levels",
        "masking",
        "best_epoch",
        "epochs_run",
        *metric_names,
        *[f"teacher_{m}" for m in teacher_metric_names],
    ]
    with open(run.result_file, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            teacher_columns = {f"teacher_{m}": row["teacher_test"][m] for m in teacher_metric_names}
            writer.writerow(
                {**row, "snr_levels": "|".join(str(s) for s in row["snr_levels"]), **row["test"], **teacher_columns}
            )


scenarios = list(CONFIG["snr_groups"].items())
logger.info("distill beat teacher -> ced-mini student on %s | %d scenarios", MIXTURE_KEY, len(scenarios))

results = [
    run_scenario(index, len(scenarios), *scenario)
    for index, scenario in enumerate(tqdm(scenarios, desc="scenarios", unit="run"), start=1)
]
write_result_csv(results)

header = f"{'scenario':<16}{'model':<10}" + "".join(f"{m:>18}" for m in metric_names)
print(header)
print("-" * len(header))
for _row in results:
    print(f"{_row['scenario']:<16}{'student':<10}" + "".join(f"{_row['test'][m]:>18.4f}" for m in metric_names))
    print(
        f"{'':<16}{'teacher':<10}"
        + "".join(f"{_row['teacher_test'].get(m, float('nan')):>18.4f}" for m in metric_names)
    )
logger.info("result saved to %s", run.result_file)

In [ ]:
def show_table(title, rows):
    width = max(len(str(key)) for key, _ in rows) + 2
    print(f"\n{title}\n" + "=" * (width + 40))
    for key, value in rows:
        print(f"{key!s:<{width}}{value}")


def millions(module):
    return sum(p.numel() for p in module.parameters()) / 1e6


ced_config = ced_pretrained.config
teacher_config = teacher.config
with torch.no_grad():
    dummy = torch.zeros(1, int(CONFIG["clip_seconds"] * CONFIG["sample_rate"]))
    teacher_fbank = teacher.frontend(dummy)
    teacher_features = teacher.backbone(teacher_fbank)
    ced_spectrogram = ced_pretrained.frontend(dummy)
    ced_features = ced_pretrained.backbone(ced_spectrogram)

show_table(
    "Pipeline",
    [
        ("distillation", "waveform -> BEATs teacher (frozen) -> 527 logits -> 36 logits by mid"),
        ("", "waveform -> CED-Mini student -> 36 logits, loss = KD(student, teacher)"),
        ("input waveform", tuple(dummy.shape)),
        ("teacher Fbank", tuple(teacher_fbank.shape)),
        ("teacher patch features", tuple(teacher_features.shape)),
        ("student log-mel", tuple(ced_spectrogram.shape)),
        ("student patch features", tuple(ced_features.shape)),
        ("logits", (1, ced_config.num_classes)),
    ],
)

show_table(
    "BEATs teacher architecture",
    [
        *teacher_config.to_dict().items(),
        ("selected classes", len(teacher.class_indices)),
        ("total parameters (M)", f"{millions(teacher):.2f}"),
    ],
)

show_table(
    "CED-Mini student architecture",
    [
        *ced_config.to_dict().items(),
        ("patch tokens N (clip)", ced_features.shape[1]),
        ("backbone parameters (M)", f"{millions(ced_pretrained.backbone):.2f}"),
        ("head parameters (M)", f"{millions(ced_pretrained.head):.3f}"),
        ("total parameters (M)", f"{millions(ced_pretrained):.2f}"),
    ],
)

show_table("Student frontend", list(ced_yaml["frontend"].items()))
show_table("Teacher frontend", list(beat_yaml["frontend"].items()))
show_table("Student training hyperparameters", list(CONFIG["ced"].to_dict().items()))
show_table(
    "Loss",
    [
        ("distillation", f"KDLoss(temperature={CONFIG['ced'].kd_temperature}), no hard labels"),
        ("selection metric", "validation macro_f1 (metrics.ClassificationMetrics)"),
    ],
)
show_table(
    "Experiment grid",
    [
        ("snr groups", CONFIG["snr_groups"]),
        ("masking", "disabled (FRCRN not used in this notebook)"),
        ("scenarios", len(CONFIG["snr_groups"])),
        ("outputs", f"{run.log_file} | {run.checkpoint_dir} | {run.result_dir}"),
    ],
)

In [ ]:
from IPython.display import Image, display

from src.utils.plots import plot_loss_curves, plot_map_curves, plot_test_metrics

snr_group_names = list(CONFIG["snr_groups"])
saved_results = {
    path.stem: json.loads(path.read_text(encoding="utf-8")) for path in sorted(run.history_dir.glob("*_nomask.json"))
}

figures = [
    plot_map_curves(saved_results, snr_group_names, run.figure_dir / "map_train_val.png"),
    plot_loss_curves(
        saved_results, snr_group_names, run.figure_dir / "train_loss.png", title="CED-Mini distillation loss"
    ),
    plot_test_metrics(saved_results, snr_group_names, run.figure_dir / "test_metrics.png"),
]
for _figure in figures:
    logger.info("figure saved: %s", _figure)
    display(Image(filename=str(_figure)))